<a href="https://colab.research.google.com/github/JacekDrzycimski/6tunnel/blob/master/Assembler_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple 8-bit Assembler in Python

This notebook implements a simple, basic assembler that translates human-readable assembly instructions into binary machine code.

### Our Simple Instruction Set Architecture (ISA)
We'll support a simple 16-bit instruction format:
- **OPCODE** (4 bits)
- **DESTINATION REGISTER** (2 bits: `R0`, `R1`, `R2`, `R3`)
- **SOURCE/IMMEDIATE** (10 bits)

### Instruction Map:
- `ADD R_dest, R_src` (Opcode: `0001`)
- `SUB R_dest, R_src` (Opcode: `0010`)
- `LOAD R_dest, value` (Opcode: `0011`)
- `STORE R_dest, value` (Opcode: `0100`)

In [1]:
import re

# Define register map
REG_MAP = {
    'R0': '00',
    'R1': '01',
    'R2': '10',
    'R3': '11'
}

# Define instruction opcode map
OP_MAP = {
    'ADD': '0001',
    'SUB': '0010',
    'LOAD': '0011',
    'STORE': '0100'
}

def int_to_bin_string(val, bits):
    """Converts an integer to a binary string of a fixed bit length."""
    val = int(val)
    # Handle negative values using two's complement
    if val < 0:
        val = (1 << bits) + val
    binary = bin(val)[2:]
    if len(binary) > bits:
        raise ValueError(f"Value {val} exceeds representation of {bits} bits")
    return binary.zfill(bits)

def assemble_instruction(line):
    # Strip comments and whitespace
    line = line.split(';')[0].strip()
    if not line:
        return None

    # Basic tokenization (splits by space, comma)
    tokens = [t for t in re.split(r'[\s,]+', line) if t]
    if not tokens:
        return None

    opcode_name = tokens[0].upper()
    if opcode_name not in OP_MAP:
        raise SyntaxError(f"Unknown opcode: {opcode_name}")

    opcode_bin = OP_MAP[opcode_name]

    # Parse Destination Register
    dest_reg = tokens[1].upper()
    if dest_reg not in REG_MAP:
        raise SyntaxError(f"Unknown destination register: {dest_reg}")
    dest_bin = REG_MAP[dest_reg]

    # Parse Source register or Immediate Value
    source_token = tokens[2].upper()
    if source_token in REG_MAP:
        # Register-to-register format. Use standard register bits + padding
        source_bin = REG_MAP[source_token].zfill(10)
    else:
        # Immediate value format
        try:
            val = int(source_token.replace('#', '')) # Support '#' symbol prefix commonly used for constants
            source_bin = int_to_bin_string(val, 10)
        except ValueError:
            raise SyntaxError(f"Invalid source or immediate value: {tokens[2]}")

    # Combine parts to form 16-bit instruction
    binary_instruction = opcode_bin + dest_bin + source_bin
    return binary_instruction

def assemble_program(program_str):
    lines = program_str.strip().split('\n')
    machine_code = []
    for idx, line in enumerate(lines):
        try:
            bin_instr = assemble_instruction(line)
            if bin_instr:
                machine_code.append((line, bin_instr))
        except Exception as e:
            print(f"Error on line {idx + 1}: '{line}' -> {e}")
            return []
    return machine_code

### Test the Assembler with custom Assembly code

In [2]:
assembly_program = """
LOAD R0, #15     ; Load constant 15 into register R0
LOAD R1, #25     ; Load constant 25 into register R1
ADD R0, R1       ; Add contents of R1 to R0
STORE R0, #255   ; Store result of R0 at address 255
"""

compiled = assemble_program(assembly_program)

print(f"{'Assembly Instruction':<30} | {'Binary Machine Code':<16} | Hex")
print("-" * 60)
for instr, bin_val in compiled:
    hex_val = hex(int(bin_val, 2))[2:].zfill(4).upper()
    print(f"{instr:<30} | {bin_val} | 0x{hex_val}")

Assembly Instruction           | Binary Machine Code | Hex
------------------------------------------------------------
LOAD R0, #15     ; Load constant 15 into register R0 | 0011000000001111 | 0x300F
LOAD R1, #25     ; Load constant 25 into register R1 | 0011010000011001 | 0x3419
ADD R0, R1       ; Add contents of R1 to R0 | 0001000000000001 | 0x1001
STORE R0, #255   ; Store result of R0 at address 255 | 0100000011111111 | 0x40FF


### Exporting to a Binary File

To write the 16-bit instructions (which are currently represented as 16-character strings of `'0'`s and `'1'`s) to a true binary file, we need to convert each 16-bit string into a 2-byte sequence (`bytes`) in either big-endian or little-endian format.

In [3]:
def write_bin_file(compiled_code, output_filepath):
    """Converts compiled binary strings to raw bytes and writes them to a file."""
    with open(output_filepath, 'wb') as f:
        for _, bin_str in compiled_code:
            # Convert 16-bit string to integer
            val = int(bin_str, 2)
            # Pack integer into 2 bytes (big-endian format 'big')
            byte_data = val.to_bytes(2, byteorder='big')
            f.write(byte_data)
    print(f"Successfully wrote {len(compiled_code) * 2} bytes to {output_filepath}")

# Define output path
output_file = 'program.bin'

# Write the compiled instructions
write_bin_file(compiled, output_file)

Successfully wrote 8 bytes to program.bin


### Verifying the Binary File contents
Let's read the binary file back to verify the bytes match our expected output.

In [4]:
with open(output_file, 'rb') as f:
    raw_bytes = f.read()

print(f"Raw bytes from file: {raw_bytes.hex().upper()}")

# Print as 16-bit segments
for i in range(0, len(raw_bytes), 2):
    word = raw_bytes[i:i+2]
    print(f"Instruction {i//2 + 1}: 0x{word.hex().upper()} ({bin(int.from_bytes(word, 'big'))[2:].zfill(16)})")

Raw bytes from file: 300F3419100140FF
Instruction 1: 0x300F (0011000000001111)
Instruction 2: 0x3419 (0011010000011001)
Instruction 3: 0x1001 (0001000000000001)
Instruction 4: 0x40FF (0100000011111111)
